In [2]:
import scanpy as sc
import scipy.io as sio
import pandas as pd

In [3]:
output_dir = "/rds/general/user/ao225/home/CardiaFinal/Data/Rogue_Ready/"
def rogueextracter(adata, name):

    sio.mmwrite(f"{output_dir}expr_{name}.mtx", adata.raw.X.T)
   
    pd.Series(adata.raw.var_names).to_csv(f"{output_dir}genes_{name}.csv", index=False)
    pd.Series(adata.obs_names).to_csv(f"{output_dir}cells_{name}.csv", index=False)
    
    meta_df = adata.obs
    meta_df.to_csv(f"{output_dir}meta_{name}.csv")

In [4]:
adata = sc.read_h5ad("/rds/general/user/ao225/home/CardiaFinal/Data/Post_Processed/Science/Cardio.h5ad")

In [5]:
adata = adata[adata.obs["disease"].isin(["normal","dilated cardiomyopathy"])]
adata = adata[adata.obs["tissue"] == "heart left ventricle"]
adata = adata[adata.obs["assay"] == "10x 3' v3"]

In [6]:
clean_obs = adata.obs
donors_per_genotype = clean_obs.groupby("Primary.Genetic.Diagnosis")["donor_id"].nunique()

print("\n--- Genotypes with More Than 3 Donors Only ---")
high_donor_genotypes = donors_per_genotype[donors_per_genotype > 3]

if not high_donor_genotypes.empty:
    print(high_donor_genotypes.reset_index(name="number_of_donors").to_string(index=False))
else:
    print("No genotypes found with more than 3 unique donors.")


--- Genotypes with More Than 3 Donors Only ---
Primary.Genetic.Diagnosis  number_of_donors
                     LMNA                11
                    PVneg                 8
                    RBM20                 7
                      TTN                12
                  control                12


/var/tmp/pbs.3677633.pbs-7/ipykernel_2700806/2681375560.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  donors_per_genotype = clean_obs.groupby("Primary.Genetic.Diagnosis")["donor_id"].nunique()


In [7]:
adata = adata[adata.obs["Primary.Genetic.Diagnosis"].isin(["LMNA", "TTN", "RBM20", "PVneg","control"])]

In [8]:
pathway_map = {
    "LMNA":  "chromatin",
    "TTN":   "sarcomere",
    "RBM20": "splicing",
    "PVneg": "PVneg",
    "control": "control"
}

In [9]:
adata.obs["Genotype"] = adata.obs["Primary.Genetic.Diagnosis"].map(pathway_map)

/var/tmp/pbs.3677633.pbs-7/ipykernel_2700806/1777082383.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["Genotype"] = adata.obs["Primary.Genetic.Diagnosis"].map(pathway_map)


In [10]:
adata.X

<91956x32383 sparse matrix of type '<class 'numpy.float32'>'
	with 205550559 stored elements in Compressed Sparse Row format>

In [11]:
adata.obs["donor_id"].nunique()

50

In [12]:
adata.obs["Genotype"].value_counts()

Genotype
control      39667
chromatin    22186
sarcomere    17965
splicing      6798
PVneg         5340
Name: count, dtype: int64

In [14]:
adata.write_h5ad("/rds/general/user/ao225/home/CardiaFinal/Data/Post_Processed/Science/LVCardio.h5ad")

In [16]:
rogueextracter(adata, "lv")

In [41]:
adatanor = adata[adata.obs["disease"] == "normal"]
adatadcm = adata[adata.obs["disease"] == "dilated cardiomyopathy"]

In [42]:
rogueextracter(adatadcm, "dcm")
rogueextracter(adatanor, "nor")

In [14]:
control = adata[adata.obs["Genotype"] == "control"]
sarcomere  = adata[adata.obs["Genotype"] == "sarcomere"]
chromatin     = adata[adata.obs["Genotype"] == "chromatin"]
splicing       = adata[adata.obs["Genotype"] == "splicing"]
PVneg          = adata[adata.obs["Genotype"] == "PVneg"]

In [15]:
rogueextracter(control, "control")
rogueextracter(sarcomere, "sarcomere")
rogueextracter(chromatin, "chromatin")
rogueextracter(splicing, "splicing")
rogueextracter(PVneg, "PVneg")